<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/TTS/speechT5_BARK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#第一类：speecht5模型
#加载speechT5模型以及processor（包含tokenizer和feature extraction）
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model: reconstructing file:   0%|          |  0.00B /  238kB            

spm_char.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  585MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
#输入的文本
inputs = processor(text="Don't count the days, make the days count.", return_tensors="pt")

In [5]:
# # 直接读取已经提前算好的 speaker embedding，供后面的 TTS 推理使用。
# 这段代码的作用是：给 TTS 模型提供“说话人的声音特征”。
# 文本 → TTS 模型 + speaker embedding → 语音
# 这里的 speaker_embeddings 就是在告诉模型：
# “请用这个人的声音特征来读这段文字。”

In [6]:
from datasets import load_dataset
import torch

# 由于原本的 .py 脚本失效且没有 dataset.json，
# 我们直接加载 Hugging Face 自动转换并托管在 parquet 分支下的数据文件
embeddings_dataset = load_dataset(
    "parquet",
    data_files="https://huggingface.co/datasets/Matthijs/cmu-arctic-xvectors/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet"
)

# 获取其中一个说话人的 X-vector 并转换为 Tensor [1, 512]
speaker_embeddings = torch.tensor(
    embeddings_dataset["train"][7306]["xvector"]
).unsqueeze(0)

model.safetensors: reconstructing file:   0%|          |  0.00B /  585MB            

model.safetensors: downloading bytes:           |  0.00B            

default/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.3MB            

default/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
#看下这个feature的形状
speaker_embeddings.shape

torch.Size([1, 512])

In [8]:
#看下inputs的情况，inputs_id即token_id
inputs

{'input_ids': tensor([[ 4, 51,  8,  9, 31,  6,  4, 17,  8, 16,  9,  6,  4,  6, 11,  5,  4, 14,
          7, 22, 12, 23,  4, 18,  7, 28,  5,  4,  6, 11,  5,  4, 14,  7, 22, 12,
          4, 17,  8, 16,  9,  6, 26,  2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [9]:
#经过模型生成logmel
spectrogram = model.generate_speech(inputs["input_ids"], speaker_embeddings)


In [10]:
#它的形状，第一维表示长度，第二维表示80-bin mel spectrograms.
spectrogram.shape

torch.Size([140, 80])

In [11]:
#加载hifi-gan的vocoder
from transformers import SpeechT5HifiGan

vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 50.7MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [12]:
#化成speech
speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

model.safetensors: reconstructing file:   0%|          |  0.00B / 50.6MB            

model.safetensors: downloading bytes:           |  0.00B            

In [13]:
#听下speech
from IPython.display import Audio

Audio(speech, rate=16000)

In [14]:
#第二类：bark模型
#加载与训练的barkmodel
from transformers import BarkModel, BarkProcessor

model = BarkModel.from_pretrained("suno/bark-small")

#这个processor里面有tokenizer和存储了speaker embeddings
processor = BarkProcessor.from_pretrained("suno/bark-small")

config.json:   0%|          | 0.00/8.80k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.68GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.68GB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [15]:
# 选一个 a speaker embedding
inputs = processor("This is a test!", voice_preset="v2/en_speaker_4")

#speech_output就是waveform
speech_output = model.generate(**inputs).cpu().numpy()

speaker_embeddings/v2/en_speaker_4_seman(…): reconstructing file:   0%|          |  0.00B / 2.42kB            

speaker_embeddings/v2/en_speaker_4_seman(…): downloading bytes:           |  0.00B            

speaker_embeddings/v2/en_speaker_4_coars(…): reconstructing file:   0%|          |  0.00B / 7.02kB            

speaker_embeddings/v2/en_speaker_4_coars(…): downloading bytes:           |  0.00B            

speaker_embeddings/v2/en_speaker_4_fine_(…): reconstructing file:   0%|          |  0.00B / 13.9kB            

speaker_embeddings/v2/en_speaker_4_fine_(…): downloading bytes:           |  0.00B            

[transformers] Passing `generation_config` together with generation-related arguments=({'min_eos_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t

In [16]:
speech_output.shape

(1, 58880)

In [17]:
#bark输出一般为24khz，听下
from IPython.display import Audio
Audio(speech_output[0], rate=24000)

In [18]:
#bark支持多语言，用中文试下
inputs = processor(
    "这是一个测试！",
    voice_preset="v2/zh_speaker_2"
)

speech_output1 = model.generate(**inputs).cpu().numpy()

speaker_embeddings/v2/zh_speaker_2_seman(…): reconstructing file:   0%|          |  0.00B / 2.15kB            

speaker_embeddings/v2/zh_speaker_2_seman(…): downloading bytes:           |  0.00B            

speaker_embeddings/v2/zh_speaker_2_coars(…): reconstructing file:   0%|          |  0.00B / 6.21kB            

speaker_embeddings/v2/zh_speaker_2_coars(…): downloading bytes:           |  0.00B            

speaker_embeddings/v2/zh_speaker_2_fine_(…): reconstructing file:   0%|          |  0.00B / 12.3kB            

speaker_embeddings/v2/zh_speaker_2_fine_(…): downloading bytes:           |  0.00B            

[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc

In [ ]:
from IPython.display import Audio
Audio(speech_output1[0], rate=24000)

In [ ]:
#bark还支持非语言的：比如laughing, sighing，crying，这里用清下嗓子：clears_throat,输入特殊格式即可
inputs = processor(
    "[clears throat] This is a test ... and I just took a long pause.",
    voice_preset="v2/fr_speaker_1",
)

speech_output2 = model.generate(**inputs).cpu().numpy()

In [ ]:
from IPython.display import Audio
Audio(speech_output2[0], rate=24000)

In [22]:
#输入多个文本一起处理
input_list = [
    "[laughing] Hello uh ..., my dog is cute [laughter]",
    "Let's try generating speech, with Bark, a text-to-speech model",
    "♪ In the jungle, the mighty jungle, the lion barks tonight ♪",
]

# also add a speaker embedding
inputs = processor(input_list, voice_preset="v2/en_speaker_3")

speech_output = model.generate(**inputs).cpu().numpy()

speaker_embeddings/v2/en_speaker_3_seman(…): reconstructing file:   0%|          |  0.00B / 3.54kB            

speaker_embeddings/v2/en_speaker_3_seman(…): downloading bytes:           |  0.00B            

speaker_embeddings/v2/en_speaker_3_coars(…): reconstructing file:   0%|          |  0.00B / 10.4kB            

speaker_embeddings/v2/en_speaker_3_coars(…): downloading bytes:           |  0.00B            

speaker_embeddings/v2/en_speaker_3_fine_(…): reconstructing file:   0%|          |  0.00B / 20.6kB            

speaker_embeddings/v2/en_speaker_3_fine_(…): downloading bytes:           |  0.00B            

[transformers] The attention mask and the pad token id were not set, with a batched input. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gen

In [23]:
#分别看下三个处理结果
from IPython.display import Audio

sampling_rate = model.generation_config.sample_rate
Audio(speech_output[0], rate=sampling_rate)

In [24]:
Audio(speech_output[1], rate=sampling_rate)

In [25]:
Audio(speech_output[2], rate=sampling_rate)